# Build a Multi-Agent AI System with Open-Source LLM & CrewAI

In this notebook, a three-agent system (Planner, Writer, Editor) collaborates to research, write, and edit a blog article on any topic, using Groq-hosted open-source LLMs and live web search.

## Setting Up

- Install CrewAI, CrewAI Tools, and LiteLLM (CrewAI's provider routing layer).

In [ ]:
!pip install -U crewai crewai-tools litellm -q

## Setting Up the LLM

- Each agent gets its own `LLM` instance so temperature can be tuned per role: the Planner and Editor stay factual (low temperature), the Writer gets more creative freedom (higher temperature).
- The API key is entered via `getpass` at runtime rather than hardcoded, so it is never saved into the notebook file.

In [ ]:
import os
from getpass import getpass
from crewai import Agent, Task, Crew, Process, LLM

# Workaround for CrewAI bug #5886: cache_breakpoint isn't stripped for Groq
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

planner_llm = LLM(model="groq/llama-3.3-70b-versatile", temperature=0.3)  # factual, focused
writer_llm  = LLM(model="groq/llama-3.3-70b-versatile", temperature=0.7)  # creative, engaging
editor_llm  = LLM(model="groq/llama-3.3-70b-versatile", temperature=0.3)  # precise, consistent

## Adding a Search Tool

- `SerperDevTool` gives the Planner access to real, current Google search results, so its research is grounded in live data rather than the LLM's training-data memory.

In [ ]:
from getpass import getpass
from crewai_tools import SerperDevTool

os.environ["SERPER_API_KEY"] = getpass("Enter your Serper API key: ")

search_tool = SerperDevTool()

## Creating Agents

- Define your Agents, and provide them a `role`, `goal`, and `backstory`.
- LLMs perform better when they are role playing.

### Agent: Planner

In [ ]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory=(
        "You are working on planning a blog article about {topic}. "
        "You collect information that helps the audience learn something "
        "and make informed decisions. Your work is the basis for the "
        "Content Writer to write an article on this topic."
    ),
    tools=[search_tool],
    llm=planner_llm,
    verbose=True
)

### Agent: Writer

In [ ]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate opinion pieces about {topic}",
    backstory=(
        "You are working on writing a new opinion piece about {topic}. "
        "You base your writing on the work of the Content Planner, who "
        "provides an outline and relevant context about the topic."
    ),
    llm=writer_llm,
    verbose=True
)

### Agent: Editor

In [ ]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with the writing style of the organization",
    backstory=(
        "You are an editor who receives a blog post from the Content Writer. "
        "Your goal is to review the blog post to ensure it follows journalistic "
        "best practices, provides balanced viewpoints, and avoids major "
        "controversial topics or opinions when possible."
    ),
    llm=editor_llm,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output`, and `agent`.

### Task: Plan

In [ ]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering their interests and pain points.\n"
        "3. Develop a detailed content outline including an introduction, key points, "
        "and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document with an outline, "
                     "audience analysis, SEO keywords, and resources.",
    agent=planner
)

### Task: Write

In [ ]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
        "3. Sections/subtitles are properly named in an engaging manner.\n"
        "4. Ensure the post is structured with an engaging introduction, "
        "insightful body, and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and alignment with the brand's voice."
    ),
    expected_output="A well-written blog post in markdown format, ready for publication, "
                     "each section should have 2 or 3 paragraphs.",
    agent=writer
)

### Task: Edit

In [ ]:
edit = Task(
    description=(
        "Proofread the given blog post for grammatical errors and alignment "
        "with the brand's voice."
    ),
    expected_output="A polished, publish-ready blog post in markdown format.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents.
- Pass the tasks to be performed by those agents.
  - **Note**: for this example, the tasks are performed sequentially (i.e. they are dependent on each other), so the *order* of the task list matters.
- `verbose=True` shows all the logs of the execution.

In [ ]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    process=Process.sequential,
    verbose=True
)

## Running the Crew

- `kickoff_async` (with `await`) is used instead of `kickoff` because Colab runs inside an active event loop, which the plain synchronous call can't handle.

In [ ]:
result = await crew.kickoff_async(inputs={"topic": "Artificial Intelligence"})
print(result)

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [ ]:
result = await crew.kickoff_async(inputs={"topic": "YOUR TOPIC HERE"})
print(result)